# Notebook 01 — Data Collection & Data Dictionary

**Maternal & Infant Mortality Risk Prediction Using Data Mining Techniques**

Course: Data Mining Techniques (DMT), B.Tech IT  
Authors: Sakthi Darshan K · Suriyaa R  
K. Ramakrishnan College of Technology (Autonomous), Trichy

---

## Purpose
1. Generate the 5 000-row **SYNTHETIC** dataset with documented risk rules.
2. Load the real **UCI Maternal Health Risk** dataset for comparison.
3. Present a data dictionary table for both datasets.

> ⚠️ **SYNTHETIC — generated for academic demonstration. Not real clinical data.**

In [ ]:
# Standard imports
import sys, os
import pandas as pd
import numpy as np

# Add project root to path so we can import src/
sys.path.insert(0, os.path.abspath('..'))

from src.data_generation import generate_synthetic_dataset, save_dataset

print('Imports OK')

## 1. Generate the Synthetic Dataset

In [ ]:
# Generate 5,000 synthetic records (seed=42 for reproducibility)
df_synth = generate_synthetic_dataset(n=5000, seed=42)

# Save to data/raw/
save_dataset(df_synth)

print(f"\nShape: {df_synth.shape}")
df_synth.head()

In [ ]:
# Class balance of the synthetic target variable
print("SYNTHETIC — Class Balance:")
print(df_synth['risk_label'].value_counts())
print(f"\nHigh-Risk %: {(df_synth['risk_label']=='HIGH-RISK').mean()*100:.1f}%")

### Documented Risk-Score Rule

The target variable is derived from a weighted, rule-based risk score:

| Condition | Points | Clinical Rationale |
|-----------|--------|--------------------|
| Severe anemia (Hb < 7 g/dL) | 4 | Life-threatening |
| Moderate anemia (Hb 7–10.9) | 2 | WHO moderate anemia |
| Hypertension (SBP 140–159) | 3 | Pre-eclampsia risk |
| Severe hypertension (SBP ≥ 160) | 5 | Eclampsia risk |
| Gestational diabetes | 2 | Macrosomia / complications |
| Severe pregnancy complications | 4 | Direct obstetric cause |
| Mild pregnancy complications | 1 | Needs monitoring |
| Low birth weight (1.5–2.5 kg) | 3 | Neonatal mortality |
| Very low birth weight (< 1.5 kg) | 5 | Extreme prematurity |
| Preterm (32–36 weeks) | 3 | Respiratory distress |
| Very preterm (< 32 weeks) | 5 | High NICU need |
| ANC visits < 4 | 2 | Below WHO minimum |
| Home delivery | 1 | No emergency backup |
| Mother age < 18 | 2 | Adolescent pregnancy |
| Mother age > 35 | 2 | Advanced maternal age |

**Threshold:** Total score ≥ 5 → HIGH-RISK.  
**Noise:** Gaussian noise N(0, 1.5) is added to the score before thresholding to inject ~5% label noise, ensuring the classification task is non-trivial.

## 2. Load the Real UCI Dataset

In [ ]:
# Download from Kaggle/UCI — using the standard CSV available at:
# https://archive.ics.uci.edu/dataset/863/maternal+health+risk
#
# If the download URL changes, place the CSV manually in data/raw/

UCI_URL = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2024/2024-10-08/maternal_health_risk.csv'
UCI_LOCAL = os.path.join('..', 'data', 'raw', 'uci_maternal_health_risk.csv')

try:
    df_uci = pd.read_csv(UCI_URL)
    df_uci.to_csv(UCI_LOCAL, index=False)
    print(f"UCI dataset downloaded → {UCI_LOCAL}")
except Exception as e:
    print(f"Download failed ({e}). Trying local copy...")
    if os.path.exists(UCI_LOCAL):
        df_uci = pd.read_csv(UCI_LOCAL)
    else:
        # Fallback: load directly from Kaggle URL
        KAGGLE_URL = 'https://raw.githubusercontent.com/CSLI-SLU/Maternal-Health-Risk-Data-Set/main/Maternal%20Health%20Risk%20Data%20Set.csv'
        df_uci = pd.read_csv(KAGGLE_URL)
        df_uci.to_csv(UCI_LOCAL, index=False)
        print(f"UCI dataset downloaded (Kaggle fallback) → {UCI_LOCAL}")

print(f"\nUCI Dataset shape: {df_uci.shape}")
df_uci.head()

In [ ]:
# Quick overview of UCI dataset
print("UCI Dataset info:")
print(df_uci.info())
print("\nUCI Risk Level distribution:")
print(df_uci['RiskLevel'].value_counts())

## 3. Data Dictionary — Synthetic Dataset (12 Attributes)

> **SYNTHETIC — generated for academic demonstration**

In [ ]:
# Data Dictionary table for the synthetic dataset
data_dict = pd.DataFrame({
    'Attribute': [
        'mother_age', 'education_level', 'family_income',
        'residence_urban_rural', 'blood_pressure_systolic',
        'blood_pressure_diastolic', 'hemoglobin_level', 'diabetes_status',
        'pregnancy_complications', 'gestational_age', 'birth_weight',
        'antenatal_care_visits', 'place_of_delivery', 'risk_label'
    ],
    'Type': [
        'Numeric (int)', 'Categorical', 'Numeric (int)',
        'Binary', 'Numeric (int)',
        'Numeric (int)', 'Numeric (float)', 'Binary',
        'Categorical', 'Numeric (int)', 'Numeric (float)',
        'Numeric (int)', 'Categorical', 'Binary (Target)'
    ],
    'Unit': [
        'Years', '—', 'INR/month',
        '—', 'mmHg',
        'mmHg', 'g/dL', '—',
        '—', 'Weeks', 'Kilograms',
        'Count', '—', '—'
    ],
    'Range': [
        '14–48', '{No Education, Primary, Secondary, Higher}', '3 000–120 000',
        '{Urban, Rural}', '80–200',
        '50–130', '5.0–17.0', '{Yes, No}',
        '{None, Mild, Severe}', '24–43', '0.80–5.00',
        '0–15', '{Hospital, Clinic, Home}', '{HIGH-RISK, LOW-RISK}'
    ],
    'Source / Rationale': [
        'NFHS-5 (median ~23)', 'NFHS-5 proportions', 'Log-normal (Indian households)',
        'NFHS-5 (~65% rural)', 'WHO normal ~120/80',
        'Correlated with SBP', 'NFHS-5 (mean ~11.8 pregnant)', 'Prevalence ~8% India',
        'Clinical categories', 'WHO term = 37–42 wks', 'WHO LBW < 2.5 kg',
        'WHO min 4; NFHS-5 median ~5', 'NFHS-5 institutional ~89%', 'Rule-based score ≥ 5'
    ],
    'Category': [
        'Maternal & Care', 'Socioeconomic', 'Socioeconomic',
        'Socioeconomic', 'Clinical',
        'Clinical', 'Clinical', 'Clinical',
        'Clinical', 'Clinical', 'Clinical',
        'Maternal & Care', 'Maternal & Care', 'Target'
    ]
})

# Display the data dictionary
data_dict.style.set_caption('SYNTHETIC Dataset — Data Dictionary')

## 4. Data Dictionary — UCI Real Dataset

In [ ]:
uci_dict = pd.DataFrame({
    'Attribute': ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate', 'RiskLevel'],
    'Type': ['Numeric', 'Numeric', 'Numeric', 'Numeric', 'Numeric', 'Numeric', 'Categorical (Target)'],
    'Unit': ['Years', 'mmHg', 'mmHg', 'mmol/L', '°F', 'bpm', '—'],
    'Range': ['10–70', '70–160', '49–100', '6–19', '98–103', '7–90', '{low risk, mid risk, high risk}'],
    'Source': ['UCI/Kaggle'] * 7,
})

uci_dict.style.set_caption('UCI Maternal Health Risk Dataset — Data Dictionary')

## 5. Descriptive Statistics

In [ ]:
# Synthetic dataset — descriptive stats
print("SYNTHETIC Dataset — Descriptive Statistics:")
df_synth.describe(include='all')

In [ ]:
# Check for missing values
print("Missing values in synthetic dataset:")
print(df_synth.isnull().sum())
print(f"\nTotal missing: {df_synth.isnull().sum().sum()}")

## Summary

- **Synthetic dataset**: 5,000 rows × 14 columns (12 input + 1 target + index) generated successfully.
- **UCI dataset**: ~1,014 rows × 7 columns downloaded for benchmarking.
- Risk score rule is documented and produces a non-trivial classification task thanks to Gaussian noise injection.
- All data saved to `data/raw/`.

**Next → Notebook 02: Preprocessing**